# Data Cleaning — SAP News Intelligence
Cleans `clean_data.json` (56 articles, 9 columns) and saves a validated `clean_data.json` ready for chunking and embedding.

**Steps covered:**
1. Load & inspect raw data
2. Normalize unicode / whitespace
3. Parse & sort dates
4. Reconstruct missing URLs
5. Deduplicate
6. Drop short/empty rows
7. Add `data_source_type` column
8. Save & validate


## 1. Imports & configuration

In [1]:
import re
import unicodedata
from pathlib import Path

import pandas as pd

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR      = Path("/home/jovyan/vault/NLPProject/AI-CEO-Strategic-Intelligence-Agent/notebook/data/")          # adjust if needed
RAW_PATH      = DATA_DIR / "clean_data.json"
OUT_PATH      = DATA_DIR / "clean_data.json"   # overwrite in-place

MIN_TEXT_LEN  = 50    # drop articles shorter than this (failed scrapes)


## 2. Load raw data

In [2]:
df = pd.read_json(RAW_PATH)
print(f"Shape:   {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head(3)


Shape:   (249, 10)
Columns: ['title', 'description', 'content', 'source', 'published', 'url', 'text', 'clean_text', 'clean_length', 'data_source_type']


,title,description,content,source,published,url,text,clean_text,clean_length,data_source_type
0,Ask HN: Industries Ripe For Disruption?,,Look at a lot of big traditional B2B businesse...,Hacker News,2009-02-10T19:54:17.000Z,https://news.ycombinator.com/item?id=475799,Ask HN: Industries Ripe For Disruption? Look a...,Ask HN: Industries Ripe For Disruption? Look a...,1119,third_party
1,Strategies to Take on an Established Competitor,,I wonder how the strategy will change if I wer...,Hacker News,2009-03-20T03:41:55.000Z,https://news.ycombinator.com/item?id=524464,Strategies to Take on an Established Competito...,Strategies to Take on an Established Competito...,188,third_party
2,"Programmers becoming ""starving artists""",,Among the many erroneous assumptions this make...,Hacker News,2010-02-21T06:22:42.000Z,https://news.ycombinator.com/item?id=1140233,"Programmers becoming ""starving artists"" Among ...","Programmers becoming ""starving artists"" Among ...",1037,third_party


In [3]:
print("─── Null counts ───")
print(df.isnull().sum())
print(f"\nDtypes:\n{df.dtypes}")


─── Null counts ───
title               0
description         0
content             0
source              0
published           0
url                 0
text                0
clean_text          0
clean_length        0
data_source_type    0
dtype: int64

Dtypes:
title               object
description         object
content             object
source              object
published           object
url                 object
text                object
clean_text          object
clean_length         int64
data_source_type    object
dtype: object


## 3. Normalize unicode & whitespace

Raw scrapes often contain:
- Non-breaking spaces (`\u00a0`), zero-width chars (`\u200b`), BOM (`\ufeff`)
- Stray HTML tags left by the scraper
- Consecutive spaces / mixed newlines


In [4]:
def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    # Canonical unicode form (converts curly quotes, ligatures, etc.)
    text = unicodedata.normalize("NFKC", text)
    # Remove invisible / control characters
    text = re.sub(r"[\u200b\u00a0\u200e\u200f\ufeff\u2028\u2029]", " ", text)
    # Strip stray HTML tags
    text = re.sub(r"<[^>]+>", " ", text)
    # Collapse all whitespace to single space
    text = re.sub(r"\s+", " ", text)
    return text.strip()


before_sample = df.loc[0, "clean_text"][:120]

df["title"]      = df["title"].map(normalize_text)
df["clean_text"] = df["clean_text"].map(normalize_text)
df["source"]     = df["source"].str.strip()
df["clean_length"] = df["clean_text"].str.len()

after_sample = df.loc[0, "clean_text"][:120]
print("Before:", repr(before_sample))
print("After: ", repr(after_sample))


Before: "Ask HN: Industries Ripe For Disruption? Look at a lot of big traditional B2B businesses. They haven't gotten much covera"
After:  "Ask HN: Industries Ripe For Disruption? Look at a lot of big traditional B2B businesses. They haven't gotten much covera"


## 4. Parse & sort dates

- SAP News articles have proper ISO 8601 timestamps
- 26 GNews articles have `null` dates (no pub date in the feed) — kept as `NaT`, not dropped


In [5]:
df["published"] = pd.to_datetime(df["published"], errors="coerce", utc=True)

n_null = int(df["published"].isna().sum())
print(f"Rows with no date (NaT): {n_null}  — sources: {df.loc[df['published'].isna(), 'source'].value_counts().to_dict()}")
print(f"Date range (non-null):   {df['published'].min()}  →  {df['published'].max()}")

# Sort chronologically; rows with no date go to the end
df = df.sort_values("published", na_position="last").reset_index(drop=True)
print(f"\nSorted: {df['published'].is_monotonic_increasing} (ignoring NaT tail)")


Rows with no date (NaT): 0  — sources: {}
Date range (non-null):   2009-02-10 19:54:17+00:00  →  2026-06-24 11:15:00+00:00

Sorted: True (ignoring NaT tail)


## 5. Reconstruct missing URLs

30 SAP News articles had no URL in the raw data.  
We reconstruct a canonical slug from the title (`https://news.sap.com/2026/<slug>/`).


In [6]:
def make_sap_slug(title: str) -> str:
    slug = title.lower()
    slug = re.sub(r"[^a-z0-9\s-]", "", slug)
    slug = re.sub(r"\s+", "-", slug).strip("-")
    return f"https://news.sap.com/2026/{slug}/"


mask_no_url = df["url"].isna() | (df["url"].astype(str).str.strip() == "")
print(f"Rows missing URL: {mask_no_url.sum()}")

df.loc[mask_no_url & (df["source"] == "SAP News"), "url"] = (
    df.loc[mask_no_url & (df["source"] == "SAP News"), "title"]
    .map(make_sap_slug)
)

print(f"Still missing after reconstruction: {df['url'].isna().sum()}")
print("\nExample reconstructed URL:")
print(df[df["source"] == "SAP News"]["url"].iloc[0])


Rows missing URL: 0
Still missing after reconstruction: 0

Example reconstructed URL:
https://news.sap.com/2026/02/sap-a-leader-2026-gartner-magic-quadrant-personalization-engines/


## 6. Deduplicate

In [7]:
before = len(df)

df = df.drop_duplicates(subset=["url"])
df = df.drop_duplicates(subset=["title", "source"])

after = len(df)
print(f"Removed {before - after} duplicate rows ({before} → {after})")


Removed 0 duplicate rows (249 → 249)


## 7. Drop short / empty rows

Any article with `clean_text` shorter than `50` chars is likely a failed scrape.

In [8]:
short_mask = df["clean_text"].str.len() < MIN_TEXT_LEN
print(f"Rows below minimum length ({MIN_TEXT_LEN} chars): {short_mask.sum()}")
df = df[~short_mask].reset_index(drop=True)
print(f"Rows remaining: {len(df)}")


Rows below minimum length (50 chars): 0
Rows remaining: 249


## 8. Add `data_source_type` column

Distinguishes SAP-owned content from third-party press coverage — useful for filtering in downstream analysis.

In [9]:
df["data_source_type"] = df["source"].apply(
    lambda s: "owned" if s == "SAP News" else "third_party"
)

print(df["data_source_type"].value_counts().to_string())


data_source_type
owned          148
third_party    101


## 9. Summary report

In [10]:
print("═" * 45)
print("  FINAL DATA CLEANING SUMMARY")
print("═" * 45)
print(f"  Total rows          : {len(df)}")
print(f"  Columns             : {len(df.columns)}")
print(f"  Missing dates (NaT) : {df['published'].isna().sum()}  (GNews — no date in feed)")
print(f"  Missing URLs        : {df['url'].isna().sum()}")
print(f"  Duplicate rows      : {df.duplicated().sum()}")
print(f"  Min clean_length    : {df['clean_length'].min()}")
print(f"  Max clean_length    : {df['clean_length'].max()}")
print(f"  Mean clean_length   : {df['clean_length'].mean():.0f}")
print(f"  Owned (SAP News)    : {(df['data_source_type']=='owned').sum()}")
print(f"  Third-party         : {(df['data_source_type']=='third_party').sum()}")
print("═" * 45)

df.head(3)


═════════════════════════════════════════════
  FINAL DATA CLEANING SUMMARY
═════════════════════════════════════════════
  Total rows          : 249
  Columns             : 10
  Missing dates (NaT) : 0  (GNews — no date in feed)
  Missing URLs        : 0
  Duplicate rows      : 0
  Min clean_length    : 137
  Max clean_length    : 45460
  Mean clean_length   : 4211
  Owned (SAP News)    : 148
  Third-party         : 101
═════════════════════════════════════════════


,title,description,content,source,published,url,text,clean_text,clean_length,data_source_type
0,Ask HN: Industries Ripe For Disruption?,,Look at a lot of big traditional B2B businesse...,Hacker News,2009-02-10 19:54:17+00:00,https://news.ycombinator.com/item?id=475799,Ask HN: Industries Ripe For Disruption? Look a...,Ask HN: Industries Ripe For Disruption? Look a...,1119,third_party
1,Strategies to Take on an Established Competitor,,I wonder how the strategy will change if I wer...,Hacker News,2009-03-20 03:41:55+00:00,https://news.ycombinator.com/item?id=524464,Strategies to Take on an Established Competito...,Strategies to Take on an Established Competito...,188,third_party
2,"Programmers becoming ""starving artists""",,Among the many erroneous assumptions this make...,Hacker News,2010-02-21 06:22:42+00:00,https://news.ycombinator.com/item?id=1140233,"Programmers becoming ""starving artists"" Among ...","Programmers becoming ""starving artists"" Among ...",1037,third_party


## 10. Save cleaned data

In [11]:
df.to_json(OUT_PATH, orient="records", indent=4, date_format="iso")
print(f"Saved → {OUT_PATH}  ({len(df)} rows, {len(df.columns)} columns)")


Saved → /home/jovyan/vault/NLPProject/AI-CEO-Strategic-Intelligence-Agent/notebook/data/clean_data.json  (249 rows, 10 columns)
